# Parcours KORYXA Formation — Statistiques & Data Science avec Python
## Projet Professionnel : Analyse, Segmentation Client & Prévision des Ventes

Ce notebook applique la démarche scientifique aux données réelles de KORYXA Distribution (2 500 clients et 730 jours de ventes).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Fixer la graine aleatoire pour la reproductibilite stricte
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Environnement Data Science KORYXA configure avec succes.')

### 1. Chargement et Audit de Qualite des Donnees
Inspection des types, verification de l'absence de valeurs manquantes et controle du grain.

In [ ]:
df_clients = pd.read_csv('transactions_clients.csv')
df_ventes = pd.read_csv('ventes_journalieres.csv')
df_ventes['date'] = pd.to_datetime(df_ventes['date'])

print(f'Dimensions Clients : {df_clients.shape}')
print(f'Dimensions Ventes : {df_ventes.shape}')
display(df_clients.head(3))

### 2. Statistiques Descriptives & Distributions
Calcul des tendances centrales (moyenne vs mediane) et dispersion (ecart-type, IQR).

In [ ]:
stats_panier = df_clients['panier_moyen_fcfa'].describe(percentiles=[0.25, 0.5, 0.75, 0.95])
iqr = stats_panier['75%'] - stats_panier['25%']
print('--- Synthese Statistique Panier Moyen (FCFA) ---')
print(f'Moyenne : {stats_panier["mean"]:,.0f} FCFA')
print(f'Mediane : {stats_panier["50%"]:,.0f} FCFA')
print(f'IQR     : {iqr:,.0f} FCFA')

plt.figure(figsize=(10, 4))
plt.hist(df_clients['panier_moyen_fcfa'] / 1000, bins=40, color='#00a86b', edgecolor='black', alpha=0.8)
plt.title('Distribution du Panier Moyen Client (kFCFA)')
plt.xlabel('Panier Moyen (kFCFA)')
plt.ylabel('Frequence (Nombre de Clients)')
plt.show()

### 3. Tests d'Hypotheses & Comparaison de Groupes
Testons si les clients avec `satisfaction >= 4.0` ont un panier moyen significativement superieur.

In [ ]:
from scipy import stats

groupe_satisfait = df_clients[df_clients['taux_satisfaction'] >= 4.0]['panier_moyen_fcfa']
groupe_neutre = df_clients[df_clients['taux_satisfaction'] < 4.0]['panier_moyen_fcfa']

# Test t de Welch (sans supposer l'egalite des variances)
stat_t, p_val = stats.ttest_ind(groupe_satisfait, groupe_neutre, equal_var=False)
print(f'Statistique t de Welch : {stat_t:.4f}')
print(f'P-valeur : {p_val:.4e}')
if p_val < 0.05:
    print('=> Difference statistiquement significative au seuil alpha = 5%')

### 4. Segmentation Client K-Means (Pipeline RFM)
Standardisation via `StandardScaler`, selection du nombre de clusters $k$ et interpretation metier.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

rfm_features = ['recence_jours', 'frequence_achats', 'montant_total_fcfa']
X = df_clients[rfm_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_clients['cluster'] = kmeans.fit_predict(X_scaled)

profil_clusters = df_clients.groupby('cluster')[rfm_features].mean()
display(profil_clusters.round(1))

### 5. Prevision Temporelle des Ventes Journalieres
Separation train/test chronologique sans fuite d'information et calcul de l'erreur MAE.

In [ ]:
# Train sur les 600 premiers jours, Test sur les 130 derniers
split_idx = 600
train = df_ventes.iloc[:split_idx]
test = df_ventes.iloc[split_idx:].copy()

# Baseline Saisonniere (valeur de la semaine precedente J-7)
test['prev_baseline_saison'] = df_ventes['ventes_fcfa'].shift(7).iloc[split_idx:]

mae_baseline = np.mean(np.abs(test['ventes_fcfa'] - test['prev_baseline_saison']))
print(f'MAE Baseline Saisonniere J-7 : {mae_baseline:,.0f} FCFA')

plt.figure(figsize=(12, 4))
plt.plot(test['date'], test['ventes_fcfa'] / 1e6, label='Reel', color='black')
plt.plot(test['date'], test['prev_baseline_saison'] / 1e6, label='Baseline Saisonniere J-7', color='#00a86b', linestyle='--')
plt.title('Prevision des Ventes Journalieres (Millions FCFA)')
plt.xlabel('Date')
plt.ylabel('Ventes (M FCFA)')
plt.legend()
plt.show()